# EDA — The evaluation datasets

Two evaluation data sources feed the study's three evaluation notebooks:

**1. MentalChat16K test split** — scores the fine-tuning itself
(`evaluate_metric` + `evaluate_llm_judge`). The prior `mhsupportchat` study
evaluated on this corpus too, with a different split:

| | prior study (mhsupportchat) | this study |
|---|---|---|
| **Source** | `ShenLab/MentalChat16K` (HuggingFace) | same corpus, pinned revision |
| **Cleaning** | dedupe only | dedupe + quality filters (placeholder / letter / >700-word rows dropped) |
| **Split** | row-level 70/20/10, seed 42 → `data/test.csv` (1,598 rows) | grouped by user text 70/20/10, seed 42 — no question can appear in two splits |
| **Evaluation set** | the full test csv | the test split, grouped to unique questions with multi-references (`evaluator.testbed()`) |

**2. Multi-turn CounselChat** (`Jingy2000/multi-turn-counsel-chat`) — the
single-vs-multi architecture evaluation (`evaluate_single_vs_multi`): real
CounselChat.com Q&A expanded into client/counselor dialogues with GPT-4.
Multi-turn is required there because the architectures only diverge with
history (verbatim vs summarizer memory). The seeded cases are previewed at
the bottom.


In [ ]:
# Setup
import os
os.environ["PYTHONUTF8"] = "1"

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))               # ft root (ft_utils, config)
sys.path.insert(0, str(Path.cwd().parent / "evaluate"))  # evaluator, single_vs_multi

import json

import matplotlib.pyplot as plt
import pandas as pd

import evaluator
import ft_utils

train_df, val_df, test_df = ft_utils.load_and_split("mentalchat16k")
bed = evaluator.testbed()
print(f"MentalChat16K splits: train {len(train_df)} / val {len(val_df)} / test {len(test_df)}")
print(f"testbed: {len(bed)} unique questions, "
      f"{(bed['refs'].str.len() > 1).sum()} with multiple references")


In [ ]:
# The prior study's evaluation csv (row-level split of the same corpus)
OLD_TEST = Path(r"C:\Users\laoli\OneDrive\Desktop\multi agent mental health chat\mhsupportchat\data\test.csv")
if OLD_TEST.exists():
    old_df = pd.read_csv(OLD_TEST)
    old_q = set(old_df["input"].astype(str).str.strip())
    print(f"prior study test.csv: {len(old_df)} rows, {len(old_q)} unique questions")
    overlap_test = old_q & set(bed["input"])
    overlap_train = old_q & set(train_df["input"])
    print(f"overlap with THIS study's test questions:  {len(overlap_test)}")
    print(f"overlap with THIS study's TRAIN questions: {len(overlap_train)} "
          "<- why scores are not comparable across the two studies")
else:
    print("prior study csv not found on this machine — comparison skipped")


In [ ]:
# Length distributions of the MentalChat16K test questions / references
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(test_df["input"].str.split().str.len(), bins=40, color="#7B5CBF")
axes[0].set_title("test questions — words")
axes[1].hist(test_df["output"].str.split().str.len(), bins=40, color="#EE7733")
axes[1].set_title("test reference answers — words")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
print(test_df["output"].str.split().str.len().describe().round(1).to_string())


In [ ]:
# Multi-turn CounselChat: the whole corpus, then the seeded replay cases
import single_vs_multi as svm

data = json.loads(svm.RAW_PATH.read_text(encoding="utf-8"))
n_client = pd.Series([sum(1 for m in d["messages"] if m["role"] == "client")
                      for d in data])
print(f"{len(data)} conversations | client turns per conversation:")
print(n_client.describe().round(1).to_string())

cases = svm.load_cases()
tasks = svm.flat_turns(cases)
print(f"\nstratified replay: {len(cases)} cases, 2 per category "
      f"(seed {ft_utils.SEED}), {len(tasks)} client turns, "
      f"cap {svm.MAX_CLIENT_TURNS}/case\n")
for c in cases:
    first = c["turns"][0]["message"] if c["turns"] else "(empty)"
    print(f"case {c['case']:>3} [{c['category']:<10}] | {len(c['turns'])} turns | {first[:70]}")


In [ ]:
# One full replay case, as both systems will see it unfold
c = cases[0]
for t in c["turns"][:3]:
    print(f"--- turn {t['turn']} (history: {len(t['history'])} exchanges) ---")
    print(f"CLIENT:    {t['message'][:160]}")
    print(f"REFERENCE: {t['reference'][:160]}")
    print()
